# AI-Based Ground Vibration Detection
**Human / Vehicle Activity Classification: Signal Processing + Machine Learning (simulation notebook)**

Pipeline simulated here:

`Synthetic vibration -> (analog stage assumed done) -> sampling at 2 kHz -> digital band-pass filter -> features (RMS, peak, dominant frequency, band energy) -> dataset.csv -> Decision Tree / Random Forest -> prediction -> C code for the ESP32`

> **Important:** all signals in this notebook are **synthetic** (generated by code). They test the pipeline. The final accuracy must come from your own recorded data.

**How to use:** run the cells from top to bottom (Kernel -> Restart & Run All). Cells are numbered in the headings.

## 1. Imports and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import butter, sosfilt, sosfiltfilt, sosfreqz

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

plt.rcParams.update({"font.size": 12})

# ---------- Sampling settings (must match the ESP32 firmware) ----------
fs = 2000                  # sampling rate in Hz
duration = 1.0             # window length in seconds
N = int(fs * duration)     # samples per window = 2000
t = np.arange(N) / fs      # time axis for one window

rng = np.random.default_rng(42)   # seeded: results are repeatable

CLASSES = ["No activity", "Human footsteps", "Light vehicle", "Other vibration"]
FEATURES = ["rms", "peak", "dominant_freq", "band_energy"]

print("fs =", fs, "Hz | Nyquist =", fs // 2, "Hz | samples per window =", N)

## 2. Synthetic signal generator

Each function makes one 1-second window for a class. Frequencies and amplitudes are **assumptions** for testing, in arbitrary units.

- **No activity:** small random noise only.
- **Footsteps:** repeated low-frequency thumps (20-40 Hz), about 1.5-2 steps per second.
- **Light vehicle:** continuous hum (25-50 Hz) plus two harmonics.
- **Other:** 1-3 sharp knocks at higher frequency (80-150 Hz).

In [ ]:
def add_burst(x, start_time, freq, tau, amp):
    """Add one decaying sine 'thump' to x, starting at start_time (seconds)."""
    start = int(start_time * fs)
    if start >= len(x):
        return
    L = min(int(0.3 * fs), len(x) - start)
    tb = np.arange(L) / fs
    x[start:start + L] += amp * np.exp(-tb / tau) * np.sin(2 * np.pi * freq * tb)

def make_no_activity():
    return rng.normal(0, 0.02, N)

def make_footsteps():
    x = rng.normal(0, 0.02, N)
    period = rng.uniform(0.5, 0.7)
    first = rng.uniform(0, period)
    for ts in np.arange(first, duration, period):
        add_burst(x, ts, freq=rng.uniform(20, 40), tau=0.05, amp=rng.uniform(0.3, 0.6))
    return x

def make_vehicle():
    f0 = rng.uniform(25, 50)
    phase = rng.uniform(0, 2 * np.pi, 3)
    x = sum((1 / k) * np.sin(2 * np.pi * k * f0 * t + phase[k - 1]) for k in (1, 2, 3))
    envelope = 1 + 0.2 * np.sin(2 * np.pi * 0.5 * t + rng.uniform(0, 2 * np.pi))
    return 0.5 * x * envelope + rng.normal(0, 0.03, N)

def make_other():
    x = rng.normal(0, 0.03, N)
    for _ in range(rng.integers(1, 4)):
        add_burst(x, rng.uniform(0, 0.8), freq=rng.uniform(80, 150), tau=0.02, amp=rng.uniform(0.6, 1.0))
    return x

GENERATORS = [make_no_activity, make_footsteps, make_vehicle, make_other]

def make_window(label):
    """One 1-second window for class number 0..3."""
    return GENERATORS[label]()

## 3. Look at the four classes (time domain and frequency domain)

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 10))
for label in range(4):
    x = make_window(label)
    axes[label, 0].plot(t, x)
    axes[label, 0].set_title(CLASSES[label] + " - time domain")
    axes[label, 0].set_ylim(-1.2, 1.2)
    axes[label, 0].set_xlabel("Time (s)"); axes[label, 0].set_ylabel("Amplitude")

    spectrum = np.abs(np.fft.rfft(x)) / N
    freqs_plot = np.fft.rfftfreq(N, 1 / fs)
    axes[label, 1].plot(freqs_plot, spectrum)
    axes[label, 1].set_xlim(0, 300)
    axes[label, 1].set_title(CLASSES[label] + " - frequency domain")
    axes[label, 1].set_xlabel("Frequency (Hz)"); axes[label, 1].set_ylabel("Magnitude")
plt.tight_layout()
plt.savefig("fig_classes.png", dpi=200)
plt.show()

## 4. Digital filter design

A **4th-order Butterworth IIR band-pass, 3-150 Hz**, designed with `scipy.signal.butter` at fs = 2000 Hz.

- Below 3 Hz it removes slow drift.
- Above 150 Hz it removes high-frequency noise (the slide says the highest signal frequency is under 150 Hz).
- Butterworth is maximally flat in the passband (Signal Processing Module 5).
- `sos` (second-order sections) is the numerically stable form, which matters on a small microcontroller.

In [ ]:
sos = butter(4, [3, 150], btype="bandpass", fs=fs, output="sos")

# Frequency response of the filter
w, h = sosfreqz(sos, worN=4096, fs=fs)
plt.figure(figsize=(8, 4))
plt.plot(w, 20 * np.log10(np.maximum(np.abs(h), 1e-6)))
plt.axvline(3, color="red", ls="--"); plt.axvline(150, color="red", ls="--")
plt.ylim(-80, 5); plt.xlim(0, 500)
plt.xlabel("Frequency (Hz)"); plt.ylabel("Gain (dB)")
plt.title("Band-pass filter response (3-150 Hz)")
plt.grid(True)
plt.savefig("fig_filter_response.png", dpi=200)
plt.show()

### 4b. Filter before / after (use this plot on the Signal Processing slide)

In [ ]:
clean = make_window(1)                                  # footsteps
drift = 0.3 * np.sin(2 * np.pi * 0.5 * t)               # slow baseline drift
hf_noise = 0.15 * np.sin(2 * np.pi * 400 * t)           # high-frequency noise
noisy = clean + drift + hf_noise
filtered = sosfiltfilt(sos, noisy)                      # zero-phase, for the plot only

freqs_plot = np.fft.rfftfreq(N, 1 / fs)
spec_before = np.abs(np.fft.rfft(noisy)) / N
spec_after = np.abs(np.fft.rfft(filtered)) / N

fig, ax = plt.subplots(2, 2, figsize=(12, 6))
ax[0, 0].plot(t, noisy);                    ax[0, 0].set_title("Before filter - time")
ax[0, 1].plot(t, filtered, color="green");  ax[0, 1].set_title("After filter - time")
ax[1, 0].plot(freqs_plot, spec_before);     ax[1, 0].set_title("Before filter - spectrum")
ax[1, 1].plot(freqs_plot, spec_after, color="green"); ax[1, 1].set_title("After filter - spectrum")
for a in ax[0]:
    a.set_xlabel("Time (s)"); a.set_ylabel("Amplitude"); a.set_ylim(-1.5, 1.5)
for a in ax[1]:
    a.set_xlabel("Frequency (Hz)"); a.set_ylabel("Magnitude"); a.set_xlim(0, 500)
    a.axvline(150, color="red", ls="--")
plt.tight_layout()
plt.savefig("filter_before_after.png", dpi=200)
plt.show()

## 5. Feature extraction

Each 2000-sample window becomes 4 numbers:

| Feature | Meaning |
|---|---|
| `rms` | overall signal energy |
| `peak` | largest swing in the window |
| `dominant_freq` | frequency of the biggest FFT peak (3 Hz and above) |
| `band_energy` | spectral energy between 20 and 60 Hz |

The **same function** must be used for training and for prediction.

In [ ]:
freqs = np.fft.rfftfreq(N, 1 / fs)

def extract_features(x):
    """One raw window (N samples) -> dict of 4 features."""
    x = x - np.mean(x)                      # remove DC offset
    x = sosfilt(sos, x)                     # digital band-pass (one direction, like the ESP32)

    rms = np.sqrt(np.mean(x ** 2))
    peak = np.max(np.abs(x))

    spectrum = np.abs(np.fft.rfft(x)) / N
    valid = freqs >= 3
    dominant_freq = freqs[valid][np.argmax(spectrum[valid])]

    band = (freqs >= 20) & (freqs <= 60)
    band_energy = np.sum(spectrum[band] ** 2)

    return {"rms": rms, "peak": peak, "dominant_freq": dominant_freq, "band_energy": band_energy}

# Quick test on one window of each class
for label in range(4):
    print(CLASSES[label], {k: round(float(v), 4) for k, v in extract_features(make_window(label)).items()})

## 6. Build the dataset (`dataset.csv`)

200 windows per class, with extra sensor noise so the task is not too easy.

In [ ]:
windows_per_class = 200
extra_noise = 0.05
rows = []

for label in range(4):
    for _ in range(windows_per_class):
        x = make_window(label) + rng.normal(0, extra_noise, N)
        feats = extract_features(x)
        feats["label"] = label
        rows.append(feats)

df = pd.DataFrame(rows)
df.to_csv("dataset.csv", index=False)

print("Dataset shape:", df.shape)
print(df["label"].value_counts().sort_index().to_string())
print()
print(df.groupby("label")[FEATURES].mean().round(4))

### 6b. Feature distributions per class

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, f in zip(axes, FEATURES):
    for label in range(4):
        ax.hist(df[df.label == label][f], bins=20, alpha=0.6, label=CLASSES[label])
    ax.set_title(f)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig("fig_feature_hist.png", dpi=200)
plt.show()

## 7. Machine learning

Train/test split is 75% / 25%, stratified so each class appears equally in both. We compare a **Decision Tree** (easy to put on the ESP32) with a **Random Forest**.

In [ ]:
data = pd.read_csv("dataset.csv")
X = data[FEATURES]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)

for name, mdl in [("Random Forest", rf), ("Decision Tree (depth 4)", dt)]:
    pred = mdl.predict(X_test)
    print("=" * 55)
    print(name, "- test accuracy:", round(accuracy_score(y_test, pred), 4))
    print(classification_report(y_test, pred, target_names=CLASSES, zero_division=0))

### 7b. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, mdl) in zip(axes, [("Random Forest", rf), ("Decision Tree", dt)]):
    cm = confusion_matrix(y_test, mdl.predict(X_test))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(name + " - confusion matrix")
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(CLASSES, rotation=30, ha="right"); ax.set_yticklabels(CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.savefig("fig_confusion.png", dpi=200)
plt.show()

### 7c. Which features matter?

In [ ]:
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
imp.plot(kind="barh", figsize=(6, 3), title="Random Forest feature importance")
plt.tight_layout()
plt.savefig("fig_importance.png", dpi=200)
plt.show()

### 7d. Cross-validation (a more honest accuracy)

One train/test split can be lucky. 5-fold cross-validation repeats it 5 times. On synthetic data it will still be high; real recordings will score lower.

In [ ]:
from sklearn.model_selection import cross_val_score
for name, mdl in [("Random Forest", rf), ("Decision Tree", dt)]:
    scores = cross_val_score(mdl, X, y, cv=5)
    print(f"{name}: mean = {scores.mean():.3f}, per fold = {np.round(scores, 3)}")

## 8. Use the trained model on a new window

This is what the final system does: take a new window, run **the same** `extract_features`, then predict.

In [ ]:
for true_label in range(4):
    x_new = make_window(true_label) + rng.normal(0, extra_noise, N)
    feats = extract_features(x_new)
    x_df = pd.DataFrame([feats])[FEATURES]
    pred = dt.predict(x_df)[0]
    print(f"True: {CLASSES[true_label]:16s} -> Predicted: {CLASSES[pred]}")

## 9. Export the Decision Tree as C code (for Wokwi / ESP32)

A single Decision Tree turns into plain `if / else` code that runs on the ESP32 with no library. Paste the printed function into your Wokwi sketch.

The function expects the four features in this order: `rms, peak, dominant_freq, band_energy`, calculated exactly like `extract_features` above.

In [ ]:
def tree_to_c(tree_model, feature_names, func_name="classify"):
    t_ = tree_model.tree_
    lines = [f"int {func_name}(float {', float '.join(feature_names)}) {{"]
    def recurse(node, depth):
        pad = "  " * depth
        if t_.feature[node] != -2:                      # internal node
            name = feature_names[t_.feature[node]]
            thr = t_.threshold[node]
            lines.append(f"{pad}if ({name} <= {thr:.6f}f) {{")
            recurse(t_.children_left[node], depth + 1)
            lines.append(f"{pad}}} else {{")
            recurse(t_.children_right[node], depth + 1)
            lines.append(f"{pad}}}")
        else:                                           # leaf
            cls = int(np.argmax(t_.value[node][0]))
            lines.append(f"{pad}return {tree_model.classes_[cls]};")
    recurse(0, 1)
    lines.append("}")
    return "\n".join(lines)

c_code = tree_to_c(dt, FEATURES)
print("// 0 = No activity, 1 = Human footsteps, 2 = Light vehicle, 3 = Other vibration")
print(c_code)

with open("classify_tree.h", "w") as f:
    f.write("// 0 = No activity, 1 = Human footsteps, 2 = Light vehicle, 3 = Other vibration\n")
    f.write(c_code + "\n")

## 10. Export test windows as C arrays (to replay in Wokwi)

Wokwi cannot read this notebook's data directly, so we paste one short window per class into the sketch as a `const` array. Rounded to 3 decimals to keep the size small. These are the **raw window values** (before the ESP32 filter), roughly centred on zero.

In [ ]:
def window_to_c_array(x, name):
    vals = ", ".join(f"{v:.3f}f" for v in x)
    return f"const float {name}[{len(x)}] = {{{vals}}};"

test_windows = {}
for label in range(4):
    x = make_window(label) + rng.normal(0, extra_noise, N)
    test_windows[label] = x
    expected = extract_features(x)
    print(f"{CLASSES[label]:16s} expected features:", {k: round(float(v), 4) for k, v in expected.items()})

with open("test_windows.h", "w") as f:
    names = ["win_none", "win_footsteps", "win_vehicle", "win_other"]
    for label, name in enumerate(names):
        f.write(window_to_c_array(test_windows[label], name) + "\n")
print("\nSaved test_windows.h (4 arrays of", N, "samples each).")
print("Compare these expected features with what the ESP32 / Wokwi computes for the same window.")

## 11. Summary and what to say in the presentation

- **Sampling:** 2000 Hz, so the Nyquist limit is 1000 Hz and the signal of interest is below 150 Hz.
- **Filter:** 4th-order Butterworth IIR band-pass, 3-150 Hz.
- **Features:** RMS, peak, dominant frequency, band energy (20-60 Hz).
- **Classes:** no activity, human footsteps, light vehicle, other vibration.
- **Models:** Decision Tree (for the ESP32) and Random Forest (for comparison).
- **Honest note:** results are on **synthetic data**. Real accuracy comes from recordings with the actual piezo and AECD circuit.

### Next steps
1. Record real labelled data with the ESP32 and the piezo circuit.
2. Replace `dataset.csv` with the real features (same 4 columns + `label`).
3. Retrain, re-export the tree, and test in Wokwi and on hardware.